In [ ]:
import os
import sys

# Add the project root (1 level up from the notebook) to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)


In [2]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from utils import download_files, read_dataframe, save_model

In [3]:
files_path = [
    './data/yellow_tripdata_2023-01.parquet',
    './data/yellow_tripdata_2023-02.parquet'
    ]
urls = [
    'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet',
    'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet'
    ]

In [4]:
download_files(files_path, urls)

File found locally at ./data/yellow_tripdata_2023-01.parquet.
File found locally at ./data/yellow_tripdata_2023-02.parquet.


In [5]:
df_train = pd.read_parquet(files_path[0])

In [6]:
# Q1
len(df_train.columns)

19

In [7]:
# Q2
df_train['duration'] = df_train['tpep_dropoff_datetime'] - df_train['tpep_pickup_datetime']
df_train['duration'] = df_train['duration'].apply(lambda td: td.total_seconds() / 60)
df_train.duration.std()

np.float64(42.59435124195458)

In [8]:
# Q3
df_filtered = df_train[(df_train.duration >= 1) & (df_train.duration <= 60)]
(df_filtered.shape[0] / df_train.shape[0]) * 100

98.1220282212598

In [9]:
# Q4
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

# Convert categorical columns to string properly
df_filtered.loc[:, categorical] = df_filtered[categorical].copy().astype('string')

train_dicts = df_filtered[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

TARGET = 'duration'
y_train = df_filtered[TARGET].values

X_train.shape[1]


/var/folders/2f/6_zp30nx2k52b5mmkn3rd14w0000gn/T/ipykernel_50663/789450167.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['161' '43' '48' ... '114' '230' '262']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_filtered.loc[:, categorical] = df_filtered[categorical].astype(str)
/var/folders/2f/6_zp30nx2k52b5mmkn3rd14w0000gn/T/ipykernel_50663/789450167.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['141' '237' '238' ... '239' '79' '143']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_filtered.loc[:, categorical] = df_filtered[categorical].astype(str)


516

In [10]:
# Q5
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)

root_mean_squared_error(y_train, y_pred)

7.658406582175197

In [11]:
# Q6
df_val = pd.read_parquet(files_path[1])
df_val['duration'] = df_val.tpep_dropoff_datetime - df_val.tpep_pickup_datetime
df_val.duration = df_val.duration.apply(lambda td: td.total_seconds() / 60)
df_val_filtered = df_val[(df_val.duration >= 1) & (df_val.duration <= 60)]
df_val_filtered.loc[:, categorical] = df_val_filtered[categorical].fillna('').astype('string')
val_dicts = df_val_filtered[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)
y_val = df_val_filtered[TARGET].values
y_pred = lr.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
rmse

/var/folders/2f/6_zp30nx2k52b5mmkn3rd14w0000gn/T/ipykernel_50663/383936345.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '<StringArray>
['142', '132', '161', '148', '137', '263',  '48', '114', '114', '125',
 ...
 '164', '256', '141', '114', '262', '249', '186', '158',  '79', '161']
Length: 2855951, dtype: string' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df_val_filtered.loc[:, categorical] = df_val_filtered[categorical].fillna('').astype('string')
/var/folders/2f/6_zp30nx2k52b5mmkn3rd14w0000gn/T/ipykernel_50663/383936345.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '<StringArray>
['163',  '26', '145', '236', '244', '141', '243', '211', '249', '107',
 ...
 '231', '261', '140', '112',  '48', '140',  '79', '143', '162', '140']
Length: 2855951, dtype: string' has dtype incompatible with int3

7.820096870991671